# 03 Sentinel-1 GRD Data Acquisition via Google Earth Engine

**Author:** Florian Klaver

In this notebook the **Sentinel-1 GRD IW** product is acquired via `COPERNICUS/S1_GRD` on Google Earth Engine. This SAR product is already pre-processed in terms of orbit corrections, thermal noise removal, calibration to sigma-naught (σ°) and terrain correction. Only a speckle filter and linear to dB conversion are applied before export.


### Workflow

1. Authenticate with GEE and define the airport AOI
2. Load `temporal_matches.csv` and derive **all S2 dates needing S1 coverage**
   - **3 dates per event**: `before_far`, `before_near`, `after`
   - `before_far + before_near` → negative (no-mowing) training samples
   - `before_near + after` → positive (mowing) training samples
3. Query GEE S1_GRD (ascending, IW, VV+VH, 2019–2023)
4. For each S2 date find the nearest S1 acquisition within ±3 days
5. Apply speckle filter + dB conversion server-side in GEE
6. Export matched S1 images to Google Drive
7. Verify downloaded files and report final event-level coverage

---
## 1. Setup & Authentication

In [4]:
import sys
import ee
import geemap
import pandas as pd
import numpy as np
import rasterio
import glob
import os
from datetime import timedelta

# Set up Google Earth Engine (GEE) credentials
GEE_PROJECT = 'satellite-mowing-detection'  

try:
    ee.Initialize(project=GEE_PROJECT)
    print('GEE initialised successfully.')
except Exception:
    print('Credentials not found, opening browser for authentication...')
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)
    print('GEE initialised successfully.')

GEE initialised successfully.


---
## 2. Configuration

In [5]:
sys.path.insert(0, os.path.abspath('..'))
from src.config import *

# Define paths and parameters
FIRE_BRIGADE_PATH = AV_FIREBRIGADE_PATH
TEMPORAL_MATCHES = TEMPORAL_MATCHES_CSV
S1_OUTPUT_DIR = S1_GRD_DIR
GDRIVE_FOLDER = 'satellite-mowing-s1'  # Google Drive folder name

os.makedirs(S1_OUTPUT_DIR, exist_ok=True)

S1_COLLECTION = 'COPERNICUS/S1_GRD'
ORBIT_PASS = 'ASCENDING'        # ascending-only for geometric consistency
INSTRUMENT_MODE = 'IW'          # Interferometric Wide Swath, 10 m resolution
EXPORT_SCALE = 10               # metres
EXPORT_CRS = 'EPSG:2056'        # Swiss LV95, same CRS as all S2 data
MATCH_TOLERANCE = 3             # max gap in days between S2 date and S1 date
SPECKLE_RADIUS = 30             # radius for speckle filter in meters
START_DATE = '2019-01-01'
END_DATE = '2023-12-31'

print('Configuration loaded.')

Configuration loaded.


---
## 3. Define Area of Interest (AOI)

THe Area of interest (AOI) is derived from the S2 reference tile bounding box. This guarantees the S1 export covers the **full study area** including all training sample locations.


In [6]:
from pyproj import Transformer

# Derive AOI from the S2 reference tile
S2_DIR = S2_SCENES_DIR
s2_files = sorted(glob.glob(os.path.join(S2_DIR, '*.tif')))
if not s2_files:
    raise FileNotFoundError(f'No S2 GeoTIFFs found in {S2_DIR}')

with rasterio.open(s2_files[0]) as s2:
    b = s2.bounds  # EPSG:2056

# GEE requires WGS84 for geometry definitions
transformer = Transformer.from_crs('EPSG:2056', 'EPSG:4326', always_xy=True)
lon_min, lat_min = transformer.transform(b.left,  b.bottom)
lon_max, lat_max = transformer.transform(b.right, b.top)

aoi_ee = ee.Geometry.BBox(lon_min, lat_min, lon_max, lat_max)

print(f'S2 reference: {os.path.basename(s2_files[0])}')
print(f'AOI (EPSG:2056): X=[{b.left:.1f}, {b.right:.1f}]  Y=[{b.bottom:.1f}, {b.top:.1f}]')
print(f'AOI (WGS84):     lon=[{lon_min:.5f}, {lon_max:.5f}]  lat=[{lat_min:.5f}, {lat_max:.5f}]')

m = geemap.Map(center=[(lat_min + lat_max) / 2, (lon_min + lon_max) / 2], zoom=13)
m.addLayer(aoi_ee, {'color': 'red'}, 'Study area AOI (S2 extent)')
m

S2 reference: 2019-03-01.tif
AOI (EPSG:2056): X=[2682012.9, 2685824.6]  Y=[1254788.2, 1260380.7]
AOI (WGS84):     lon=[8.52587, 8.57747]  lat=[47.43878, 47.48859]


Map(center=[47.46368296392474, 8.55166676143091], controls=(WidgetControl(options=['position', 'transparent_bg…

---
## 4. Derive All Required S1 Dates from Temporal Matches

The training pipeline uses a **3-image temporal triplet** per mowing event:

| Role | Column | Used for |
|---|---|---|
| `before_far` ($t_{n-2}$) | `before_far_file` | Reference image for **negative** (no-mowing) samples |
| `before_near` ($t_{n-1}$) | `before_near_file` | Used in **both** positive and negative sample pairs |
| `after` ($t_{n+1}$) | `after_file` | Post-mowing image for **positive** (mowing) samples |

Images of all three dates are necessary.

In [7]:
matches_df = pd.read_csv(TEMPORAL_MATCHES)
matches_df['event_date'] = pd.to_datetime(matches_df['event_date'])

# Extract S2 dates from the filenames in the CSV columns
def parse_s2_dates(col):
    return pd.to_datetime(matches_df[col].str.replace('.tif', '', regex=False))

before_far_dates = parse_s2_dates('before_far_file')
before_near_dates = parse_s2_dates('before_near_file')
after_dates = parse_s2_dates('after_file')

# Union of all three sets: every S2 date that needs a corresponding S1 image
required_s2_dates = (
    pd.concat([before_far_dates, before_near_dates, after_dates])
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

print(f'Events in temporal_matches.csv:        {len(matches_df)}')
print(f'Unique before_far dates:               {before_far_dates.nunique()}')
print(f'Unique before_near dates:              {before_near_dates.nunique()}')
print(f'Unique after dates:                    {after_dates.nunique()}')
print(f'Unique S2 dates requiring S1 coverage: {len(required_s2_dates)}')
print(f'Date range: {required_s2_dates.min().date()} to {required_s2_dates.max().date()}')

Events in temporal_matches.csv:        92
Unique before_far dates:               62
Unique before_near dates:              58
Unique after dates:                    59
Unique S2 dates requiring S1 coverage: 111
Date range: 2019-04-30 to 2023-09-16


---
## 5. Query S1 Collection and Find Nearest Acquisitions

Fetch the full S1 catalogue over the AOI (metadata only, no pixel data yet), then for each required S2 date find the nearest S1 acquisition within ±3 days.

In [8]:
# Query collection metadata only (no pixel download at this stage)
s1_collection = (
    ee.ImageCollection(S1_COLLECTION)
    .filterBounds(aoi_ee)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.eq('instrumentMode', INSTRUMENT_MODE))
    .filter(ee.Filter.eq('orbitProperties_pass', ORBIT_PASS))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
    .select(['VV', 'VH'])
)

n_images = s1_collection.size().getInfo()
print(f'Total S1 ascending IW VV+VH images over AOI (2019-2023): {n_images}')

print('Fetching image dates from GEE...')
image_list = s1_collection.toList(s1_collection.size())
s1_dates_info = []

for i in range(n_images):
    img = ee.Image(image_list.get(i))
    date_ms = img.date().getInfo()['value']  # milliseconds since epoch
    date = pd.to_datetime(date_ms, unit='ms').normalize()
    s1_dates_info.append({'index': i, 'date': date, 'date_str': date.strftime('%Y%m%d')})

s1_dates_df = pd.DataFrame(s1_dates_info)
print(f'Done. S1 images retrieved: {len(s1_dates_df)}, unique dates: {s1_dates_df["date"].nunique()}')
print(s1_dates_df.head())

Total S1 ascending IW VV+VH images over AOI (2019-2023): 471
Fetching image dates from GEE...
Done. S1 images retrieved: 471, unique dates: 471
   index       date  date_str
0      0 2019-01-01  20190101
1      1 2019-01-06  20190106
2      2 2019-01-13  20190113
3      3 2019-01-18  20190118
4      4 2019-01-25  20190125


In [9]:
# Match each required S2 date to the nearest S1 date within MATCH_TOLERANCE days
date_match_records = []

for s2_date in required_s2_dates:
    window = s1_dates_df[
        (s1_dates_df['date'] >= s2_date - timedelta(days=MATCH_TOLERANCE)) &
        (s1_dates_df['date'] <= s2_date + timedelta(days=MATCH_TOLERANCE))
    ]
    if len(window) == 0:
        date_match_records.append({
            's2_date': s2_date, 's1_date': pd.NaT,
            's1_date_str': None, 'delta_days': None, 'matched': False
        })
    else:
        idx_closest = (window['date'] - s2_date).abs().idxmin()
        best = window.loc[idx_closest]
        delta = int((best['date'] - s2_date).days)
        date_match_records.append({
            's2_date': s2_date, 's1_date': best['date'],
            's1_date_str': best['date_str'], 'delta_days': delta, 'matched': True
        })

date_match_df = pd.DataFrame(date_match_records)
matched = date_match_df[date_match_df['matched']]
unmatched = date_match_df[~date_match_df['matched']]

print(f'=== S1 Date Matching Coverage ===')
print(f'Required S2 dates:           {len(date_match_df)}')
print(f'Matched (within +/-{MATCH_TOLERANCE}d):   {len(matched)}  ({len(matched)/len(date_match_df)*100:.0f}%)')
print(f'Unmatched:                   {len(unmatched)}')
if len(unmatched) > 0:
    print('\nUnmatched S2 dates (no S1 within tolerance):')
    print(unmatched[['s2_date']].to_string(index=False))
if len(matched) > 0:
    print('\nDelta days distribution:')
    print(matched['delta_days'].value_counts().sort_index())

=== S1 Date Matching Coverage ===
Required S2 dates:           111
Matched (within +/-3d):   111  (100%)
Unmatched:                   0

Delta days distribution:
delta_days
-3     3
-2    15
-1    22
 0    37
 1    17
 2    16
 3     1
Name: count, dtype: int64


In [10]:
# Event-level coverage: how many events have all 3 S1 dates matched?
s2_to_s1 = date_match_df.set_index('s2_date')['s1_date']

event_coverage = []
for _, row in matches_df.iterrows():
    bf_date = pd.to_datetime(row['before_far_file'].replace('.tif', ''))
    bn_date = pd.to_datetime(row['before_near_file'].replace('.tif', ''))
    af_date = pd.to_datetime(row['after_file'].replace('.tif', ''))

    bf_s1 = s2_to_s1.get(bf_date, pd.NaT)
    bn_s1 = s2_to_s1.get(bn_date, pd.NaT)
    af_s1 = s2_to_s1.get(af_date, pd.NaT)

    pos_ok = pd.notna(bn_s1) and pd.notna(af_s1)   # positive pair covered
    neg_ok = pd.notna(bf_s1) and pd.notna(bn_s1)   # negative pair covered
    full_ok = pd.notna(bf_s1) and pd.notna(bn_s1) and pd.notna(af_s1)

    event_coverage.append({
        'event_date_str': row['event_date_str'],
        'bf_s2': bf_date, 'bf_s1': bf_s1,
        'bn_s2': bn_date, 'bn_s1': bn_s1,
        'af_s2': af_date, 'af_s1': af_s1,
        'pos_pair_ok': pos_ok,
        'neg_pair_ok': neg_ok,
        'full_triplet_ok': full_ok,
    })

event_cov_df = pd.DataFrame(event_coverage)
total = len(event_cov_df)

print('=== Event-Level S1 Coverage ===')
print(f'Full triplet (all 3 S1 dates matched):  {event_cov_df["full_triplet_ok"].sum()} / {total}')
print(f'Positive pair only (bn + af matched):   {event_cov_df["pos_pair_ok"].sum()} / {total}')
print(f'Negative pair only (bf + bn matched):   {event_cov_df["neg_pair_ok"].sum()} / {total}')
print()
missing = event_cov_df[~event_cov_df['full_triplet_ok']]
if len(missing) > 0:
    print('Events missing at least one S1 date:')
    print(missing[['event_date_str','bf_s2','bf_s1','bn_s2','bn_s1','af_s2','af_s1']].to_string(index=False))

=== Event-Level S1 Coverage ===
Full triplet (all 3 S1 dates matched):  92 / 92
Positive pair only (bn + af matched):   92 / 92
Negative pair only (bf + bn matched):   92 / 92



In [11]:
# Save matching tables for use in notebook 04
date_match_df.to_csv(S1_DATE_MATCHES_CSV, index=False)
event_cov_df.to_csv(S1_EVENT_COVERAGE_CSV, index=False)
print('Saved: data/s1_date_matches.csv  (s2_date -> s1_date mapping)')
print('Saved: data/s1_event_coverage.csv  (per-event triplet coverage)')

Saved: data/s1_date_matches.csv  (s2_date -> s1_date mapping)
Saved: data/s1_event_coverage.csv  (per-event triplet coverage)


---
## 6. GEE Processing Functions

Two operations applied server-side before export.

### Step 1: Speckle filter (in linear scale)
SAR images have **speckle**, grainy noise from constructive/destructive interference of radar waves from multiple sub-pixel scatterers. A **3×3 boxcar (mean) filter** is applied in linear scale. The filter must run in linear scale, averaging in dB/log space distorts the statistics.

### Step 2: Linear → dB (after filtering)
Convert sigma-naught: `10 × log₁₀(linear)`. In dB, change features like `vv_diff` represent relative changes in a physically meaningful, level-independent way.

In [12]:
def db_to_linear(image):
    """Convert dB to linear power: 10^(dB/10)"""
    return ee.Image(10.0).pow(image.divide(10.0))

def linear_to_db(image):
    """Convert linear power to dB: 10 * log10(linear)"""
    return image.log10().multiply(10.0)

def apply_speckle_filter(image):
    """Apply speckle filter in linear scale."""
    # Image is in dB, so convert to linear first
    linear_img = db_to_linear(image)
    
    # Apply focal mean filter in linear scale
    filtered_linear = linear_img.focal_mean(radius=30, kernelType='square', units='meters')
    
    # Convert back to dB and copy properties
    filtered_db = linear_to_db(filtered_linear)
    
    return ee.Image(filtered_db.copyProperties(image, ['system:time_start']))

def prepare_s1_image(image):
    """Prepare the S1 image and clip it to the study area."""
    return apply_speckle_filter(image).clip(aoi_ee)


print('Processing functions defined.')

Processing functions defined.


---
## 7. Export Matched S1 Images to Google Drive

For each unique S1 date, the processing pipeline is applied and a 2-band GeoTIFF (Band 1 = VV dB, Band 2 = VH dB) is exported to a Google Drive folder.


In [ ]:
# De-duplicate: many S2 dates map to the same S1 date
unique_s1 = (
    matched
    .drop_duplicates(subset='s1_date_str')[['s1_date', 's1_date_str']]
    .reset_index(drop=True)
)
print(f'Unique S1 images to export: {len(unique_s1)}')
print(f'(Fewer than {len(required_s2_dates)} required S2 dates because many share the same S1 image.)')

tasks = []
skipped = 0

for _, row in unique_s1.iterrows():
    s1_date = pd.Timestamp(row['s1_date'])  # ensure Timestamp type
    date_str = row['s1_date_str']             # 'YYYYMMDD'
    export_id = f'S1_{date_str}'

    # Skip if already downloaded locally (allows re-running cell safely)
    local_path = os.path.join(S1_OUTPUT_DIR, f'{export_id}.tif')
    if os.path.exists(local_path):
        skipped += 1
        continue

    # filterDate is start-inclusive, end-exclusive.
    # Using (day, day+1) gets ONLY images from that exact date.
    date_start = s1_date.strftime('%Y-%m-%d')
    date_end = (s1_date + timedelta(days=1)).strftime('%Y-%m-%d')

    img_col = (
        ee.ImageCollection(S1_COLLECTION)
        .filterBounds(aoi_ee)
        .filterDate(date_start, date_end)
        .filter(ee.Filter.eq('instrumentMode', INSTRUMENT_MODE))
        .filter(ee.Filter.eq('orbitProperties_pass', ORBIT_PASS))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
        .select(['VV', 'VH'])
        .sort('system:time_start')
    )

    # Use .mosaic() to merge multiple tiles if necessary, and copy properties from the first image to preserve metadata
    base_image = ee.Image(img_col.mosaic().copyProperties(img_col.first(), ['system:time_start']))
    image = prepare_s1_image(base_image)


    task = ee.batch.Export.image.toDrive(
        image = image,
        description = export_id,
        folder = GDRIVE_FOLDER,
        fileNamePrefix = export_id,
        scale = EXPORT_SCALE,
        crs = EXPORT_CRS,
        region = aoi_ee,                # already a BBox, covers full study area
        fileFormat = 'GeoTIFF',
        maxPixels = int(1e9),           # must be int, not float
    )
    task.start()
    tasks.append({'date_str': date_str, 'task': task})

print(f'Export tasks started:      {len(tasks)}')
print(f'Already on disk (skipped): {skipped}')
print()
print('Monitor at: https://code.earthengine.google.com/')

Unique S1 images to export: 94
(Fewer than 111 required S2 dates because many share the same S1 image.)
Export tasks started:      94
Already on disk (skipped): 0

Monitor at: https://code.earthengine.google.com/ (Tasks tab)


---
## 8. Monitor Export Status


In [16]:
if not tasks:
    print('No tasks started this session (all files on disk, or cell 7 not run yet).')
else:
    status_counts = {}
    for t in tasks:
        state = t['task'].status()['state']
        status_counts[state] = status_counts.get(state, 0) + 1

    print('=== Export Status ===')
    for state, count in sorted(status_counts.items()):
        print(f'  {state}: {count}')

    failed = [t for t in tasks if t['task'].status()['state'] == 'FAILED']
    if failed:
        print(f'\nFailed tasks ({len(failed)}):')
        for t in failed:
            msg = t['task'].status().get('error_message', 'no message')
            print(f"  S1_{t['date_str']}: {msg}")

=== Export Status ===
  COMPLETED: 94


---
## 9. Verify Downloaded Files

This cell verifies the downloaded files 

In [29]:
s1_files = sorted(glob.glob(os.path.join(S1_OUTPUT_DIR, 'S1_*.tif')))
print(f'S1 files found locally: {len(s1_files)}')
print(f'Expected:               {len(unique_s1)}')
print()

if s1_files:
    with rasterio.open(s1_files[0]) as src:
        print(f'--- Spot check: {os.path.basename(s1_files[0])} ---')
        print(f'CRS:        {src.crs}  (expected: EPSG:2056)')
        print(f'Resolution: {src.res}  (expected: (10.0, 10.0))')
        print(f'Bands:      {src.count}  (expected: 2 - VV, VH)')
        print(f'Shape:      {src.height} x {src.width} pixels')
        vv = src.read(1).astype(float)
        vh = src.read(2).astype(float)
        valid = np.isfinite(vv)
        if valid.any():
            print(f'VV range:   {vv[valid].min():.1f} to {vv[valid].max():.1f} dB')
            print(f'VH range:   {vh[np.isfinite(vh)].min():.1f} to {vh[np.isfinite(vh)].max():.1f} dB')
            print()
            print('Typical dB ranges for airport grassland:')
            print('  VV: -10 to -5 dB  (surface/soil scattering)')
            print('  VH: -20 to -12 dB (volume scattering, always lower than VV)')

S1 files found locally: 94
Expected:               94

--- Spot check: S1_20190430.tif ---
CRS:        EPSG:2056  (expected: EPSG:2056)
Resolution: (10.0, 10.0)  (expected: (10.0, 10.0))
Bands:      2  (expected: 2 - VV, VH)
Shape:      561 x 398 pixels
VV range:   -23.3 to 21.1 dB
VH range:   -30.7 to 8.8 dB

Typical dB ranges for airport grassland:
  VV: -10 to -5 dB  (surface/soil scattering)
  VH: -20 to -12 dB (volume scattering, always lower than VV)


In [30]:
# Check which events have all 3 S1 files on disk
downloaded = {os.path.basename(f).replace('.tif', '') for f in s1_files}

def s1_file_exists(s1_date):
    if pd.isna(s1_date):
        return False
    return 'S1_' + pd.Timestamp(s1_date).strftime('%Y%m%d') in downloaded

event_cov_df['bf_s1_on_disk'] = event_cov_df['bf_s1'].apply(s1_file_exists)
event_cov_df['bn_s1_on_disk'] = event_cov_df['bn_s1'].apply(s1_file_exists)
event_cov_df['af_s1_on_disk'] = event_cov_df['af_s1'].apply(s1_file_exists)
event_cov_df['fully_ready']   = (
    event_cov_df['bf_s1_on_disk'] &
    event_cov_df['bn_s1_on_disk'] &
    event_cov_df['af_s1_on_disk']
)

n_ready = event_cov_df['fully_ready'].sum()
total   = len(event_cov_df)
print(f'Events fully ready for fusion training: {n_ready} / {total}  ({n_ready/total*100:.0f}%)')

if n_ready < total:
    print('\nEvents NOT yet ready (missing at least one S1 file):')
    cols = ['event_date_str', 'bf_s1_on_disk', 'bn_s1_on_disk', 'af_s1_on_disk']
    print(event_cov_df[~event_cov_df['fully_ready']][cols].to_string(index=False))

event_cov_df.to_csv(S1_EVENT_COVERAGE_CSV, index=False)
print('\nUpdated coverage table saved to data/s1_event_coverage.csv')

Events fully ready for fusion training: 92 / 92  (100%)

Updated coverage table saved to data/s1_event_coverage.csv


---
## Additional GRD Acquisition for SNR Correction

The coherence SNR correction in notebook 06b requires GRD backscatter at the exact SLC acquisition dates (t1, t2, t3, t4 from `slc_scene_index.csv`). Those dates differ from the S2-driven dates downloaded above. The cell below downloads only the **15 SLC dates** that have no existing GRD within ±3 days.

**Prerequisites:** run cells 1–7 above first so that `ee`, `aoi_ee`, `GDRIVE_FOLDER`, `S1_OUTPUT_DIR`, `S1_COLLECTION`, `INSTRUMENT_MODE`, `ORBIT_PASS`, `EXPORT_SCALE`, `EXPORT_CRS`, and `prepare_s1_image` are all defined.

In [ ]:
# ---------------------------------------------------------------------------
# Additional GRD acquisition for SNR correction
# Hardcoded list of SLC dates with no GRD within +-3 days (derived from
# slc_scene_index.csv vs. data/Sentinel_S1/ on 2026-06-20).
# ---------------------------------------------------------------------------

SNR_MISSING_DATES = [
    '2019-07-30',
    '2019-10-10',
    '2020-07-12',
    '2020-08-29',
    '2020-10-22',
    '2020-10-28',
    '2021-06-07',
    '2021-06-25',
    '2021-08-06',
    '2021-08-24',
    '2021-09-11',
    '2023-07-27',
    '2023-08-08',
    '2023-09-25',
    '2023-10-07',
]

print("=== Additional GRD Download for SNR Correction ===")
print(f"Target dates: {len(SNR_MISSING_DATES)}\n")

snr_tasks = []
snr_skipped = 0

for date_str in SNR_MISSING_DATES:
    s1_date = pd.Timestamp(date_str)
    date_yyyymmdd = s1_date.strftime('%Y%m%d')
    export_id = f'S1_{date_yyyymmdd}'
    local_path = os.path.join(S1_OUTPUT_DIR, f'{export_id}.tif')

    if os.path.exists(local_path):
        print(f"  {date_str} -> SKIP (already on disk: {export_id}.tif)")
        snr_skipped += 1
        continue

    date_start = s1_date.strftime('%Y-%m-%d')
    date_end   = (s1_date + timedelta(days=1)).strftime('%Y-%m-%d')

    img_col = (
        ee.ImageCollection(S1_COLLECTION)
        .filterBounds(aoi_ee)
        .filterDate(date_start, date_end)
        .filter(ee.Filter.eq('instrumentMode', INSTRUMENT_MODE))
        .filter(ee.Filter.eq('orbitProperties_pass', ORBIT_PASS))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
        .select(['VV', 'VH'])
        .sort('system:time_start')
    )

    n = img_col.size().getInfo()
    if n == 0:
        print(f"  {date_str} -> WARNING: no S1 GRD image found in GEE for this date, skipping")
        continue

    base_image = ee.Image(img_col.mosaic().copyProperties(img_col.first(), ['system:time_start']))
    image = prepare_s1_image(base_image)

    task = ee.batch.Export.image.toDrive(
        image          = image,
        description    = export_id,
        folder         = GDRIVE_FOLDER,
        fileNamePrefix = export_id,
        scale          = EXPORT_SCALE,
        crs            = EXPORT_CRS,
        region         = aoi_ee,
        fileFormat     = 'GeoTIFF',
        maxPixels      = int(1e9),
    )
    task.start()
    snr_tasks.append({'date': date_str, 'export_id': export_id, 'task': task})
    print(f"  {date_str} -> export task started: {export_id}")

print(f"\nExport tasks started:      {len(snr_tasks)}")
print(f"Already on disk (skipped): {snr_skipped}")
if snr_tasks:
    print("\nMonitor progress at: https://code.earthengine.google.com/ (Tasks tab)")
    print("Download completed files from Google Drive folder 'satellite-mowing-s1' to data/Sentinel_S1/")

=== Additional GRD Download for SNR Correction ===
Target dates: 15

  2019-07-30 -> export task started: S1_20190730
  2019-10-10 -> export task started: S1_20191010
  2020-07-12 -> export task started: S1_20200712
  2020-08-29 -> export task started: S1_20200829
  2020-10-22 -> export task started: S1_20201022
  2020-10-28 -> export task started: S1_20201028
  2021-06-07 -> export task started: S1_20210607
  2021-06-25 -> export task started: S1_20210625
  2021-08-06 -> export task started: S1_20210806
  2021-08-24 -> export task started: S1_20210824
  2021-09-11 -> export task started: S1_20210911
  2023-07-27 -> export task started: S1_20230727
  2023-08-08 -> export task started: S1_20230808
  2023-09-25 -> export task started: S1_20230925
  2023-10-07 -> export task started: S1_20231007

Export tasks started:      15
Already on disk (skipped): 0

Monitor progress at: https://code.earthengine.google.com/ (Tasks tab)
Download completed files from Google Drive folder 'satellite-mowi